# Piece-wise Cox proportional hazard model

- Fit model on all data (for regression task).

- Hyperparameter tuning:

    + *Algorithm*: Bayesian optimization  - supported by *Optuna* library.

    + *Objective*: maximize C-index.

    + *Validating*: cross-validation.

- Draw survival curves to compare among values of categorical variables

## 1. Dataset Preparation
### 1.1 Import dataset

In [1]:
import sys
import os
import pandas as pd
import optuna
import warnings
import datetime

from sklearn.model_selection import StratifiedKFold
from tqdm import tqdm

dir_path = os.path.abspath(os.path.join(os.getcwd(), "..", "data"))
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

from src.model import *
from src.metric import *

In [2]:
tdp_path = os.path.join(dir_path, 'time-dependent-variables-MissForest.csv')
idp_path = os.path.join(dir_path, 'time-independent-variables-MissForest.csv')

tdp_df = pd.read_csv(tdp_path, usecols=["iso_code", "total_tests_per_thousand", 
                                                "people_fully_vaccinated_per_hundred", 
                                                "stringency_index", "status", "days"])

idp_df = pd.read_csv(idp_path)
idp_df = idp_df.drop(columns=["location", "Overall_score"], axis=0)

tdp_df.info()
idp_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 156613 entries, 0 to 156612
Data columns (total 6 columns):
 #   Column                               Non-Null Count   Dtype  
---  ------                               --------------   -----  
 0   iso_code                             156613 non-null  object 
 1   total_tests_per_thousand             156613 non-null  float64
 2   people_fully_vaccinated_per_hundred  156613 non-null  float64
 3   stringency_index                     156613 non-null  float64
 4   days                                 156613 non-null  int64  
 5   status                               156613 non-null  bool   
dtypes: bool(1), float64(3), int64(1), object(1)
memory usage: 6.1+ MB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 232 entries, 0 to 231
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   iso_code        232 non-null    object 
 1   continent       232 non-null    object 

In [3]:
idp_df['duration'] = idp_df['duration'] + 1
tdp_df['days'] = tdp_df['days'] - 1

### 1.2 Dataset pre-processing

In [4]:


# deal with datetime
def convertDate(row):
    try:
        year, month, day  = row.split('-')
        return datetime.datetime(int(year), int(month), int(day))
    except:
        return row
    
# deal with category data
def autoCategory(df: pd.DataFrame, cols: list):
    for col in cols:
        items = df[col].unique()
        for idx, item in enumerate(items):
            if idx == len(items) - 1:
                break
            df[f"{col}_{item}"] = df[col].apply(lambda x: 1 if x == item else 0)

    return df.drop(columns=cols, axis=1)

# convert first_case to days from min first_case
def convertFirstDate(row):
    return row["first_case"] - min_date

idp_df["first_case"] = idp_df["first_case"].apply(convertDate)
min_date = idp_df["first_case"].unique().min()
idp_df["start_date"] = (idp_df["first_case"] - min_date).dt.days
idp_df = idp_df.drop(columns=["first_case"], axis=0)

dummy_cols = [x for x in idp_df.columns if idp_df[x].dtypes == "object" and x not in ["iso_code", "location"]]
idp_df = autoCategory(idp_df, dummy_cols)
idp_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 232 entries, 0 to 231
Data columns (total 17 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   iso_code                            232 non-null    object 
 1   population                          232 non-null    float64
 2   median_age                          232 non-null    float64
 3   duration                            232 non-null    int64  
 4   status                              232 non-null    bool   
 5   start_date                          232 non-null    int64  
 6   continent_North America             232 non-null    int64  
 7   continent_Asia                      232 non-null    int64  
 8   continent_Africa                    232 non-null    int64  
 9   continent_Europe                    232 non-null    int64  
 10  continent_South America             232 non-null    int64  
 11  classification_HIGH-INCOME          232 non-n

In [ ]:
def model_kfold_split(timedp_df: pd.DataFrame, indp_df: pd.DataFrame, n_splits: int, random_state: int):
    """
    Return: tuple of idp list of folds + tdp list of folds
    """
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    
    idp_folds = []
    tdp_folds = []

    X = indp_df
    y = indp_df["status"]

    for train_index, test_index in skf.split(X, y):
        indp_train = X.iloc[train_index].copy()
        indp_test = X.iloc[test_index].copy()

        # Get the unique iso_codes for training
        train_iso_list = indp_train["iso_code"].drop_duplicates().tolist()

        # Mark the time series data for training or testing
        timedp_df["train"] = timedp_df["iso_code"].isin(train_iso_list)
        timedp_train = timedp_df[timedp_df["train"] == True].copy().drop(columns=["train"])
        timedp_test = timedp_df[timedp_df["train"] == False].copy().drop(columns=["train"])

        idp_folds.append((indp_train, indp_test))
        tdp_folds.append((timedp_train, timedp_test))
    
    return idp_folds, tdp_folds

## 2. Train-tuning model

### 2.1 Setup aggregate for grouping dataset

In [6]:
agg_functions = dict()
agg_functions["total_tests_per_thousand"] = 'max'
agg_functions["people_fully_vaccinated_per_hundred"] = 'max'
agg_functions["stringency_index"] = 'mean'
agg_functions["days"] = "min"
agg_functions["status"] = "max"

### 2.2 Tuning model

In [7]:
stop_days = sorted(idp_df['duration'].tolist())
len(stop_days)

232

In [ ]:
warnings.filterwarnings('ignore')

def trainModel(tdp_data: pd.DataFrame, idp_data: pd.DataFrame,
               var: float = 20, n_trials: int = 10, k_folds: int = 5) -> optuna.study.Study:
    def objective(trial: optuna.trial.Trial):
        num_of_stages = trial.suggest_int("stages", 3, 6)

        l1_penalized = trial.suggest_int("l1_penalized", 0, 100, 1) * 0.01
        l2_penalized = trial.suggest_int("l2_penalized", 0, 100, 1) * 0.01

        mock = len(stop_days) / num_of_stages
        variance = min(var, mock / 2)
        cutting_points = []

        for idx in range(1, num_of_stages):
            x = trial.suggest_int(f"mock_{idx}", stop_days[int(mock*idx-variance)], stop_days[int(mock*idx+variance)])
            cutting_points.append(x)

        idp_folds, tdp_folds = model_kfold_split(tdp_data, idp_data, n_splits=k_folds, random_state=2025)

        c_index_lst = []

        try:
            for (indp_train, indp_val), (tdp_train, tdp_val) in tqdm(zip(idp_folds, tdp_folds)):
                model, scaler = fitting_model(tdp_train, indp_train, agg_functions, cutting_points, 
                                    l2_penalized, l1_penalized, tdp_effect=True, normalized=True)
                
                valid_cp_df, _ = format_data(tdp_val, indp_val, agg_functions, cutting_points, 
                                                   tdp_effect=True, scaler=scaler, normalized=True)
                
                c_index_lst.append(C_index_calculation(model, valid_cp_df))
        except:
            print(f"error when computing, hence return 0")
            return 0

        return sum(c_index_lst) / k_folds
    
    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=n_trials)

    return study

study = trainModel(tdp_df.copy(), idp_df.copy(), 20, n_trials=20)
study.best_params

[I 2025-06-02 15:57:56,215] A new study created in memory with name: no-name-c659d129-bc7c-4a9a-ad30-82cf2294314d
5it [00:19,  3.81s/it]
[I 2025-06-02 15:58:15,366] Trial 0 finished with value: 0.6491978609625668 and parameters: {'stages': 5, 'l1_penalized': 63, 'l2_penalized': 99, 'mock_1': 224, 'mock_2': 390, 'mock_3': 581, 'mock_4': 808}. Best is trial 0 with value: 0.6491978609625668.
5it [00:18,  3.61s/it]
[I 2025-06-02 15:58:33,544] Trial 1 finished with value: 0.6534759358288771 and parameters: {'stages': 4, 'l1_penalized': 84, 'l2_penalized': 71, 'mock_1': 230, 'mock_2': 497, 'mock_3': 796}. Best is trial 1 with value: 0.6534759358288771.
5it [00:16,  3.38s/it]
[I 2025-06-02 15:58:50,538] Trial 2 finished with value: 0.6827094474153298 and parameters: {'stages': 3, 'l1_penalized': 21, 'l2_penalized': 35, 'mock_1': 289, 'mock_2': 1269}. Best is trial 2 with value: 0.6827094474153298.
5it [00:17,  3.53s/it]
[I 2025-06-02 15:59:08,321] Trial 3 finished with value: 0.75008912655971

error when computing, hence return 0


5it [00:15,  3.10s/it]
[I 2025-06-02 16:01:26,049] Trial 11 finished with value: 0.7101604278074867 and parameters: {'stages': 3, 'l1_penalized': 2, 'l2_penalized': 25, 'mock_1': 251, 'mock_2': 864}. Best is trial 3 with value: 0.7500891265597147.
5it [00:16,  3.35s/it]
[I 2025-06-02 16:01:42,938] Trial 12 finished with value: 0.6602495543672015 and parameters: {'stages': 3, 'l1_penalized': 44, 'l2_penalized': 20, 'mock_1': 251, 'mock_2': 883}. Best is trial 3 with value: 0.7500891265597147.
5it [00:16,  3.23s/it]
[I 2025-06-02 16:01:59,193] Trial 13 finished with value: 0.7108734402852048 and parameters: {'stages': 4, 'l1_penalized': 0, 'l2_penalized': 19, 'mock_1': 245, 'mock_2': 547, 'mock_3': 1445}. Best is trial 3 with value: 0.7500891265597147.
5it [00:18,  3.66s/it]
[I 2025-06-02 16:02:17,617] Trial 14 finished with value: 0.6823529411764706 and parameters: {'stages': 4, 'l1_penalized': 60, 'l2_penalized': 9, 'mock_1': 244, 'mock_2': 450, 'mock_3': 1506}. Best is trial 3 with va

error when computing, hence return 0


5it [00:17,  3.50s/it]
[I 2025-06-02 16:03:17,566] Trial 18 finished with value: 0.7194295900178254 and parameters: {'stages': 3, 'l1_penalized': 10, 'l2_penalized': 13, 'mock_1': 256, 'mock_2': 1191}. Best is trial 3 with value: 0.7500891265597147.
5it [00:17,  3.51s/it]
[I 2025-06-02 16:03:35,246] Trial 19 finished with value: 0.5704099821746881 and parameters: {'stages': 3, 'l1_penalized': 35, 'l2_penalized': 72, 'mock_1': 256, 'mock_2': 1252}. Best is trial 3 with value: 0.7500891265597147.


{'stages': 3,
 'l1_penalized': 56,
 'l2_penalized': 5,
 'mock_1': 240,
 'mock_2': 1030}

### 2.2 Train model

In [ ]:
print(f"study.best_params = {study.best_params}")

l1_penalized = study.best_params['l1_penalized'] * 0.01
l2_penalized = study.best_params['l2_penalized'] * 0.01

cutting_points = [study.best_params[f'mock_{i}'] for i in range(1, study.best_params['stages'])]
cutting_points

study.best_params = {'stages': 3, 'l1_penalized': 56, 'l2_penalized': 5, 'mock_1': 240, 'mock_2': 1030}


[240, 1030]

In [8]:
l1_penalized = 0.56
l2_penalized = 0.05
cutting_points = [240, 1030]

In [ ]:
model, scaler = fitting_model(tdp_df.copy(), idp_df.copy(), agg_functions, cutting_points, l2_penalized, l1_penalized, 
                                       tdp_effect=True, normalized=True)

cp_df, _ = format_data(tdp_df.copy(), idp_df.copy(), agg_functions, cutting_points, 
                                     tdp_effect=True, normalized=True, scaler=scaler)

c_index_test = C_index_calculation(model, cp_df)

model.print_summary()
print(f"C_index_test of model = {c_index_test}")

<lifelines.CoxTimeVaryingFitter: fitted with 470 periods, 232 subjects, 170 events>
         event col = 'status'
         penalizer = 0.05
number of subjects = 232
 number of periods = 470
  number of events = 170
partial log-likelihood = -721.44
  time fit was run = 2026-07-08 15:29:19 UTC

---
                                               coef exp(coef)  se(coef)  coef lower 95%  coef upper 95% exp(coef) lower 95% exp(coef) upper 95%
covariate                                                                                                                                      
classification_HIGH-INCOME                     0.18      1.19      0.10           -0.01            0.37                0.99                1.44
classification_LOW-INCOME                     -0.10      0.90      0.11           -0.33            0.12                0.72                1.13
classification_LOWER-MIDDLE INCOME            -0.11      0.90      0.11           -0.32            0.11                0.72                1.12
continent_Africa                              -0.41      0.67      0.13           -0.65           -0.16                0.52                0.85
continent_Asia                                -0.00      1.00      0.00           -0.00            0.00                1.00                1.00
continent_Europe                               0.05      1.05      0.08           -0.11            0.22                0.89                1.24
continent_North America                       -0.00      1.00      0.00           -0.00            0.00                1.00                1.00
continent_South America                        0.02      1.02      0.08           -0.13            0.17                0.88                1.19
group_Authoritarian                           -0.20      0.82      0.10           -0.39           -0.01                0.67                0.99
group_Full democracy                           0.00      1.00      0.00           -0.00            0.00                1.00                1.00
group_Hybrid regime                           -0.00      1.00      0.00           -0.00            0.00                1.00                1.00
median_age                                     0.63      1.87      0.13            0.38            0.87                1.46                2.40
people_fully_vaccinated_per_hundred::strata_1  0.02      1.02      0.16           -0.30            0.34                0.74                1.40
people_fully_vaccinated_per_hundred::strata_2 -0.69      0.50      0.10           -0.89           -0.48                0.41                0.62
people_fully_vaccinated_per_hundred::strata_3  0.00      1.00      0.00           -0.00            0.00                1.00                1.00
population                                    -0.00      1.00      0.00           -0.00            0.00                1.00                1.00
start_date                                     0.60      1.82      0.14            0.32            0.87                1.38                2.39
stringency_index::strata_1                     0.00      1.00      0.00           -0.00            0.00                1.00                1.00
stringency_index::strata_2                     0.34      1.40      0.18           -0.02            0.69                0.98                2.00
stringency_index::strata_3                     0.00      1.00      0.00           -0.00            0.00                1.00                1.00
total_tests_per_thousand::strata_1             0.02      1.02      0.14           -0.26            0.30                0.77                1.35
total_tests_per_thousand::strata_2             0.03      1.03      0.08           -0.13            0.19                0.88                1.21
total_tests_per_thousand::strata_3             0.12      1.13      0.09           -0.05            0.29                0.95                1.34

                                               cmp to     z      p  -log2(p)
covariate               

C_index_test of model = 0.7605986773407588
